In [2]:
import cutlass
import cutlass.cute as cute


In [5]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, message="CUDA_TOOLKIT_PATH environment variable is not set.*")

In [2]:
@cute.kernel
def hello_kernel():
    tidx, _, _ = cute.arch.thread_idx()
    if tidx == 0:
        cute.printf("Hello from GPU")

@cute.jit
def hello_world():
    cutlass.cuda.initialize_cuda_context()
    hello_kernel().launch(grid=(1, 1, 1), block=(32, 1, 1))

In [ ]:
compiled = cute.compile(hello_world)
compiled()

/home/warmonkeys/miniconda3/envs/cutedsl/lib/python3.12/site-packages/nvidia_cutlass_dsl/python_packages/cutlass/base_dsl/dsl.py:412: UserWarning: CUDA_TOOLKIT_PATH environment variable is not set. Cannot set toolkitPath.
  warnings.warn(message, UserWarning)


0

Hello from GPU


In [4]:
@cute.jit
def print_demo(a: cutlass.Int32, b: cutlass.Constexpr[int]):
    print("static a:", a)   # => ? (dynamic)
    print("static b:", b)   # => 2
    cute.printf("dynamic a: {}", a)
    cute.printf("dynamic b: {}", b)
    layout = cute.make_layout((a, b))
    print("static layout:", layout)       # (?,2):(1,?)
    cute.printf("dynamic layout: {}", layout)  # (8,2):(1,8)

In [5]:
print_demo(cutlass.Int32(8), 2)

static a: ?
static b: 2
static layout: (?,2):(1,?)
dynamic a: 8
dynamic b: 2
dynamic layout: (8,2):(1,8)


In [6]:
@cute.jit
def dtypes():
    a = cutlass.Int32(42)
    b = a.to(cutlass.Float32)
    c = b + 0.5
    d = c.to(cutlass.Int32)
    cute.printf("a={}, b={}, c={}, d={}", a, b, c, d)

dtypes()

a=42, b=42.000000, c=42.500000, d=42
